# K5k2 vs K5 on `nexar_val`, whole videos: per-video curves + onset-aligned mean

Illustration only (s2024, n = 1), **decides nothing**. The top-k line is closed (`KIM_RUN.md` §26.1); this does not reopen it. **K5k2** = K5 + `loss.mil_topk_pct=10` (k = 2 of 20 rows instead of 4), both trained on Nexar, so `nexar_val` is **in-domain**. **Caveat:** on Nexar the event sits mid-video, and a ruler that only measures distance to the middle scores 0.9735 whole-video macro, so these curves show **shape only**.

K5's per-frame Nexar scores already exist (`outputs/kim_gate/eval/nexar/K5`, §20, reproduced §13). K5k2 was read on DoTA / DADA / TAD / DAD only, so this notebook scores it on `nexar_val` with the **same command** §20 used for K5, then draws the two together. `nexar_test` is never touched.

Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-4 | setup, preflight (K5k2 J10, config diff vs K5 = `mil_topk_pct` only) | no | 2 min |
| 5 | `nexar_eval --save-scores` on K5k2 -> `outputs/kim/eval/nexar/K5k2` | yes | 10-20 min |
| 6 | `zs_curves`: grid + onset + one PNG per video -> `outputs/kim/REPORTS/figures/train_nexar/nexar/k5_vs_k5k2_s2024` | no | 2-3 min |

A re-run skips what is on Drive. Thesis and Thesis-V2 are only read; everything new goes to Thesis-Final.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, ckpts
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar caches, census, A2 Nexar stats, Nexar_window
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5 / K5k2 runs, K5's Nexar scores; every output here
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert 'ZS_CURVES_NEXAR_SET' in (REPO / 'core/constants.py').read_text(), 'this upload has no nexar curve set -- re-upload kat-vad/'
assert '--save-scores' in (REPO / 'core/tools/nexar_eval.py').read_text(), 'nexar_eval has no --save-scores -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5, K5K2 = constants.KIM_MAIN_ARM, constants.KIM_TOPK_ARM   # 'K5', 'K5k2'
SEED     = constants.KIM_SEED                                # 2024
ENC      = constants.V5_ENCODER                              # V2-S
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL                   # never nexar_test
NX       = constants.ZS_CURVES_NEXAR_SET
TOPK_KEY = 'loss.mil_topk_pct'
TOPK_PCT = {K5: constants.DADA_ORIGIN_MIL_TOPK_PCT, K5K2: constants.KIM_TOPK_MIL_PCT}   # 5 -> k=4, 10 -> k=2
LABEL    = {K5: f'{constants.ZS_CURVES_MAIN_LABEL} k=4 ({K5})',
            K5K2: f'{constants.ZS_CURVES_MAIN_LABEL} k=2 ({K5K2})'}
RANK_BY  = LABEL[K5K2]                                       # rank 1 = K5k2's worst per-video AUC

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
NEXAR_U  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'            # K5's and K5k2's input statistics
WINDOW   = Path(DRIVE_V2) / 'data' / f'{constants.NEXAR_DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'

KIM      = constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME          # Thesis-Final/outputs/kim
RUN      = {a: KIM / a / f's{SEED}' for a in (K5, K5K2)}
CKPT     = RUN[K5K2] / 'checkpoint_last.pt'
K5_EVAL  = constants.OUTPUT_ROOT / constants.KIM_GATE_OUTPUT_DIRNAME / 'eval' / NX / K5   # §20's K5 scores
NEW      = KIM / 'eval' / NX / K5K2                                       # this notebook
TAG      = 'k5_vs_k5k2_s2024'
FIGS     = KIM / 'REPORTS' / constants.ZS_CURVES_FIGURES_DIRNAME / 'train_nexar' / NX / TAG
LOCAL    = Path('/content/k5k2_nexar_curves')
LOCAL.mkdir(parents=True, exist_ok=True)

INPUTS = (('K5k2 checkpoint', CKPT), ('K5k2 verified', RUN[K5K2] / 'verified.json'),
          ('K5 verified', RUN[K5] / 'verified.json'),
          ('K5k2 config', RUN[K5K2] / 'config.yaml'), ('K5 config', RUN[K5] / 'config.yaml'),
          ('A2 Nexar stats', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME),
          ('Nexar census', CENSUS), ('Nexar CLIP s1', NEXAR_X), ('Nexar V2-S s1', NEXAR_U), ('Nexar defs', WINDOW),
          ('K5 Nexar read (§20)', K5_EVAL / 'nexar_readout.json'), ('K5 Nexar scores (§20)', K5_EVAL / 'scores'))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | scores -> {NEW} | figures -> {FIGS}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 5 needs one'

In [ ]:
import torch
import yaml

from core.data.v2_splits import load_split      # noqa: E402
from core.tools.nexar_eval import READOUT_JSON  # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def stage(src, dst, ids):
    '''``ids`` (+ json/npz manifests) of a Drive dir to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    dst.mkdir(parents=True, exist_ok=True)
    for v in ids:
        if not (dst / f'{v}.npy').exists():
            shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
    for pattern in ('*.json', '*.npz'):
        for m in src.glob(pattern):
            shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def flat(d, prefix=''):
    out = {}
    for k, v in d.items():
        key = f'{prefix}{k}'
        out.update(flat(v, key + '.') if isinstance(v, dict) else {key: v})
    return out


def headline(r):
    '''Whole-video and crop numbers of one nexar_readout.json (printed, decide nothing).'''
    return {'whole macro': r['whole']['macro']['mean'], 'whole micro': r['whole']['micro'],
            'whole AP': r['whole']['micro_ap'], 'crops macro': r['crops']['mean_over_placements']['macro']['mean']}


# Preflight on small files only (C39)
steps = {a: json.loads((RUN[a] / 'verified.json').read_text())['step'] for a in (K5, K5K2)}
assert steps[K5] == steps[K5K2], f'K5 and K5k2 trained for different steps: {steps}'
STEPS = steps[K5K2]
rec = json.loads((RUN[K5K2] / 'verified.json').read_text())
assert rec['ckpt_bytes'] == CKPT.stat().st_size, f'K5k2 checkpoint is not the verified one: {rec}'
assert metrics_last(RUN[K5K2] / 'metrics.jsonl')['global_step'] == STEPS, 'K5k2: J10'
cfg = {a: flat(yaml.safe_load((RUN[a] / 'config.yaml').read_text())) for a in (K5, K5K2)}
changed = sorted(k for k in cfg[K5] if k in cfg[K5K2] and cfg[K5][k] != cfg[K5K2][k])
assert changed == [TOPK_KEY], f'K5k2 differs from K5 in more than {TOPK_KEY}: {changed}'
assert (cfg[K5][TOPK_KEY], cfg[K5K2][TOPK_KEY]) == (TOPK_PCT[K5], TOPK_PCT[K5K2]), (cfg[K5][TOPK_KEY], cfg[K5K2][TOPK_KEY])
added = {k: cfg[K5K2][k] for k in cfg[K5K2] if k not in cfg[K5]}
assert all(v in (False, 'none', None) for v in added.values()), f'K5k2 has new keys that are ON: {added}'
man = json.loads((A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
assert (man['crn'], man['motion']) == (constants.V2_OFF, ENC), f'A2 Nexar stats: {man}'
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
for name, d in (('Nexar CLIP', NEXAR_X), ('Nexar V2-S', NEXAR_U)):
    miss = [v for v in NEXAR_IDS if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} of {name} missing: {miss[:5]}'
K5_IDS = sorted(p.stem for p in (K5_EVAL / 'scores').glob('*.npz'))
assert K5_IDS == NEXAR_IDS, f'K5 scores are not nexar_val: {len(K5_IDS)} files'
K5_READ = headline(json.loads((K5_EVAL / READOUT_JSON).read_text()))
print(f'preflight OK: step {STEPS} both | config diff = {TOPK_KEY} {TOPK_PCT[K5]} -> {TOPK_PCT[K5K2]} '
      f'(+ off keys {sorted(added)}) | {NEXAR_SPLIT} {len(NEXAR_IDS)} videos | K5 scores {len(K5_IDS)}')

## 5. Score K5k2 on `nexar_val` (whole videos, per-frame scores saved): §20's command for K5, K5k2's checkpoint

In [ ]:
done = (NEW / READOUT_JSON).exists() and len(list((NEW / 'scores').glob('*.npz'))) == len(NEXAR_IDS)
if not done:
    assert torch.cuda.is_available(), 'torch sees no GPU -- nexar_eval would silently score on CPU'
    l_x = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
    l_u = stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', NEXAR_IDS)
    l_stats = stage(A2_NEXAR, LOCAL / 'stage' / 'nexar_stats', [])     # manifest + statistics only
    l_defs = LOCAL / 'stage' / 'nexar_defs'
    l_defs.mkdir(parents=True, exist_ok=True)
    for f in WINDOW.glob('*.json'):
        shutil.copy2(f, l_defs / f.name)
    local = LOCAL / 'eval' / K5K2
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.nexar_eval', '--run', f's{SEED}', CKPT, '--stats-dir', l_stats,
         '--video-s1-dir', l_u, '--census', CENSUS, '--clip-s1-dir', l_x, '--data-dir', l_defs,
         '--split', NEXAR_SPLIT, '--save-scores', '--out-dir', local], log=LOCAL / 'nexar.log')
    publish(local, NEW)
    flush_drive()
r = json.loads((NEW / READOUT_JSON).read_text())
assert r['split'] == NEXAR_SPLIT and r['videos'] == len(NEXAR_IDS), (r['split'], r['videos'])
assert r['global_steps'] == {f's{SEED}': STEPS}, f"K5k2 read is not step {STEPS}: {r['global_steps']}"
assert Path(r['checkpoints'][f's{SEED}']).parent.parent.name == K5K2, f"scored the wrong checkpoint: {r['checkpoints']}"
K5K2_READ = headline(r)
print('printed, decides nothing (in-domain, n = 1, middle ruler 0.9735):')
for k in K5_READ:
    print(f'  {k:12s} K5 {K5_READ[k]:.4f} | K5k2 {K5K2_READ[k]:.4f} | K5k2 - K5 {K5K2_READ[k] - K5_READ[k]:+.4f}')

## 6. Draw: grid + onset-aligned mean (both models, GT share on the right axis), then one PNG per video

Rank 1 = K5k2's worst per-video AUC; all-normal videos last. Blue = K5k2 (k = 2), orange = K5 (k = 4).

In [ ]:
local = LOCAL / 'figures'
shutil.rmtree(local, ignore_errors=True)
reads = ['--read', NX, LABEL[K5K2], NEW, '--read', NX, LABEL[K5], K5_EVAL]
common = ['--rank-by', RANK_BY, '--name', TAG, '--out-dir', local, '--seed-label', f's{SEED}, n = 1',
          '--domain', constants.ZS_CURVES_DOMAIN_IN]
run([sys.executable, '-m', 'core.tools.zs_curves', *common, *reads])            # grid_nexar.png + onset_aligned.png
run([sys.executable, '-m', 'core.tools.zs_curves', '--all', *common, *reads])   # videos/nexar/<rank>_<id>.png
publish(local, FIGS)
t = json.loads((local / f'{constants.ZS_CURVES_STEM}_{TAG}.json').read_text())['sets'][NX]
print(f"{t['videos']} videos, {t['two_class']} two-class | median per-video AUC: "
      + ', '.join(f'{m} {a:.3f}' for m, a in t['median_video_auc'].items()))
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f'Drive flushed -- figures in {FIGS}')

In [ ]:
from IPython.display import Image, display  # noqa: E402

for name in ('onset_aligned.png', f'grid_{NX}.png'):
    display(Image(filename=str(local / name)))

## Redraw locally (no GPU)

Download `Thesis-Final/outputs/kim/eval/nexar/K5k2/` to `outputs/v_final/kim/eval/nexar/K5k2/` (K5's scores are already local under `outputs/v_final/kim_gate/eval/nexar/K5/`), then:

```bash
source .venv/bin/activate
OUT=outputs/v_final/kim/REPORTS/figures/train_nexar/nexar/k5_vs_k5k2_s2024
R=(--read nexar "LaGoVAD+KIM k=2 (K5k2)" outputs/v_final/kim/eval/nexar/K5k2
   --read nexar "LaGoVAD+KIM k=4 (K5)"   outputs/v_final/kim_gate/eval/nexar/K5)
C=(--rank-by "LaGoVAD+KIM k=2 (K5k2)" --name k5_vs_k5k2_s2024 --out-dir $OUT --domain in-domain)
python -m core.tools.zs_curves "${C[@]}" "${R[@]}"          # grid + onset
python -m core.tools.zs_curves --all "${C[@]}" "${R[@]}"    # one PNG per video
```